This notebook develops a computational pipeline to identify quoted speakers in news articles.
Speaker identification is necessary to analyze whose voices are represented in coverage of the Global North vs Global South


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_trf

import pandas as pd
import re
import spacy
from tqdm import tqdm

tqdm.pandas()

nlp = spacy.load("en_core_web_trf")

  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 23.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 653.4/653.4 kB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.3/772.3 kB 19.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [spacy]m24/25 [spacy]tion]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 21.4 MB/s  0:00:20:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 702.9/702.9 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf] [en-core-web-trf]
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [3]:
df = pd.read_csv("data_with_country_labels.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10361, 5)


,article,highlights,id,country,global_north
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4,united kingdom,Yes
1,"BAGHDAD, Iraq (CNN) -- Dressed in a Superman s...","Parents beam with pride, can't stop from smili...",a1ebb8bb4d370a1fdf28769206d572be60642d70,iraq,No
2,"BAGHDAD, Iraq (CNN) -- The women are too afrai...","Aid workers: Violence, increased cost of livin...",7c0e61ac829a3b3b653e2e3e7536cc4881d1f264,iraq,No
3,"BOGOTA, Colombia (CNN) -- A key rebel commande...",Tomas Medina Caracas was a fugitive from a U.S...,f0d73bdab711763e745cdc75850861c9018f235d,colombia,No
4,"LONDON, England (CNN) -- A chronology of bombi...",Two cars loaded with gasoline and nails found ...,85f55a3e0dd672857aaaaa80954934a57b7a2fbf,united kingdom,Yes


## Methodology

### Step 1 – Quote Extraction

We extract direct quotations using regular expressions that identify text enclosed in quotation marks.

Limitations:
- Only captures direct quotes
- Does not capture indirect speech

In [5]:
def extract_quotes(text):
    if pd.isna(text):
        return []
    quotes = re.findall(r'“(.*?)”|"(.*?)"', text)
    quotes = [q[0] if q[0] != '' else q[1] for q in quotes]
    return quotes

df["quotes"] = df["article"].progress_apply(extract_quotes)
df["num_quotes"] = df["quotes"].apply(len)

100%|██████████| 10361/10361 [00:00<00:00, 28844.65it/s]


### Step 2 – Named Entity Recognition (NER)

We use spaCy’s transformer-based English model (`en_core_web_trf`) to identify PERSON entities.

This allows us to detect candidate speaker names within each article.

In [6]:
def extract_persons(text):
    if pd.isna(text):
        return []
    doc = nlp(text)
    persons = list(set([ent.text for ent in doc.ents if ent.label_ == "PERSON"]))
    return persons

df["persons"] = df["article"].progress_apply(extract_persons)

df[["persons"]].head()

100%|██████████| 10361/10361 [1:36:30<00:00,  1.79it/s]


,persons
0,"[Londoner, Potter, Radcliffe, Rudyard Kipling,..."
1,"[Wayne Drash, Peter Grossman, Zainab, Arwa Dam..."
2,"[Mohammed, Rahim, Basma Rahim, Suha, Karima, Y..."
3,"[Tomas Medina Caracas, Juan Manuel Santos, El ..."
4,"[Margaret Thatcher, David Copeland, Abdel Bass..."


In [11]:
# Inspect random sample
sample = df.sample(5)[["article", "persons"]]
sample

,article,persons
4570,"Tunis, Tunisia (CNN) -- World powers meeting F...","[Joe Sterling, Carla Haddad Mardini, al-Assad,..."
4618,Hong Kong (CNN) -- A faulty thermometer is lik...,"[Junichi Matsumoto, Yukio Edano, Michael Fried..."
7863,"MEXICO CITY, Mexico (CNN) -- CNN chief medical...","[John Martin, Sanjay Gupta, Gupta, Sanjay]"
7495,"Cairo, Egypt (CNN) -- Islam Lotfy may look lik...","[Mubarak, Lotfy, Islam Lotfy, Cairene, Mohamed..."
952,"NEW DELHI, India (CNN) -- A Pakistani man usin...","[Kasab, Mohammed Ajmal Kasab, Sabahuddin Ahmed..."


### Step 3 – Rule-Based Speaker Attribution

To attribute quotes to speakers, we apply rule-based pattern matching.

We search for reporting verbs such as:
- said
- told
- according to

Example patterns:
- "X said"
- "according to X"

This approach prioritizes interpretability over complexity.

In [21]:
def find_speakers(text):
    if pd.isna(text):
        return []
    
    speakers = set()
    
    # Pattern 1: John Smith said
    pattern1 = r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)+)\s(?:said|told|added|according to)'
    
    # Pattern 2: said John Smith
    pattern2 = r'(?:said|told|added)\s([A-Z][a-z]+(?:\s[A-Z][a-z]+)+)'
    
    matches1 = re.findall(pattern1, text)
    matches2 = re.findall(pattern2, text)
    
    for m in matches1 + matches2:
        speakers.add(m)
    
    return list(speakers)

In [22]:
print("Average quotes per article:", df["num_quotes"].mean())

print("Articles with at least one detected speaker:",
      (df["detected_speakers"].apply(len) > 0).mean())

Average quotes per article: 7.922208281053952
Articles with at least one detected speaker: 0.4331628221214168


In [23]:
# Inspect random sample
sample = df.sample(5)[["article", "quotes", "detected_speakers"]]
sample

,article,quotes,detected_speakers
1688,"ATLANTA, Georgia (CNN) -- Boris Kodjoe owns a ...","[When I'm opening the door of my own house, so...",[Michael Schaffer]
3957,"Beirut, Lebanon (CNN) -- Seven Estonian cyclis...",[The main thing now is for our seven fellow co...,"[Marwan Charbel, Urmas Paet, Lebanese Army]"
2949,"London, England (CNN) -- The debut album by Ne...","[album of the decade, Is This It, Up the Brack...",[]
1717,"NAIROBI, Kenya (CNN) -- Those most responsible...",[Kenya will be a world example on managing vio...,[]
3797,"Madrid, Spain (CNN) -- A total of 37 former Cu...",[],[]
